# ISOM 835 · Session 8 — Gradient Boosting & the Tabular Frontier
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 9 · Prof. Hasan Arslan**

Fit the residuals, again and again: `HistGradientBoosting`, LightGBM, XGBoost, early stopping — and the tabular foundation models (TabPFN-2.5, TabICLv2) that since 2025 match tuned boosting on small data with zero tuning.

> **Frame the prediction (Hotel Bookings).** *Unit:* one reservation · *Target:* `is_canceled` · *Horizon:* at booking · *Decision:* overbooking & deposit policy.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np, time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import roc_auc_score, log_loss

URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/hotel_bookings.csv'
raw = pd.read_csv(URL)
LEAKS = ['reservation_status', 'reservation_status_date', 'assigned_room_type']
X = raw.drop(columns=['is_canceled'] + LEAKS)
for c in X.select_dtypes(object): X[c] = X[c].astype('category')      # native categorical support
y = raw['is_canceled']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)
print(X_tr.shape, X_te.shape)

## 1. Boosting in one picture: fit the residuals
Start with a constant. Fit a tiny tree to the residuals. Add it, scaled by a learning rate. Repeat. Bias falls with every tree.

In [ ]:
rng = np.random.default_rng(835); xs = np.sort(rng.uniform(0, 1, 120)); ys = np.sin(2 * np.pi * xs) + rng.normal(0, 0.3, 120)
F = np.full_like(ys, ys.mean()); lr = 0.3; grid = np.linspace(0, 1, 300); Fg = np.full_like(grid, ys.mean())
fig, axes = plt.subplots(1, 4, figsize=(14, 3)); snaps = {1: 0, 5: 1, 20: 2, 100: 3}
for t in range(1, 101):
    tree = DecisionTreeRegressor(max_depth=2, random_state=t).fit(xs[:, None], ys - F)     # fit the residuals
    F += lr * tree.predict(xs[:, None]); Fg += lr * tree.predict(grid[:, None])
    if t in snaps: ax = axes[snaps[t]]; ax.scatter(xs, ys, s=8, color='#7c8cff', alpha=0.6); ax.plot(grid, Fg, color='#2ee6c5', lw=2); ax.set_title(f'after {t} trees · MSE {np.mean((ys-F)**2):.3f}')
plt.tight_layout(); plt.show()

## 2. `HistGradientBoostingClassifier` with early stopping
Histogram binning, native missing values, `categorical_features='from_dtype'`, and early stopping on an internal validation split. The stopping iteration is where validation loss bottomed out.

In [ ]:
t0 = time.time()
gbm = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=3000, max_leaf_nodes=31, early_stopping=True,
                                     validation_fraction=0.15, n_iter_no_change=50, categorical_features='from_dtype', random_state=835).fit(X_tr, y_tr)
p = gbm.predict_proba(X_te)[:, 1]
print(f'trees kept: {gbm.n_iter_}   test AUC {roc_auc_score(y_te, p):.4f}   log-loss {log_loss(y_te, p):.4f}   ({time.time()-t0:.1f}s)')
plt.figure(figsize=(7, 3.2)); plt.plot(-np.array(gbm.train_score_), color='#2ee6c5', label='train loss'); plt.plot(-np.array(gbm.validation_score_), color='#ff6b8b', label='validation loss'); plt.xlabel('trees'); plt.ylabel('log-loss'); plt.legend(); plt.title('early stopping watches the coral line'); plt.show()

## 3. The knobs that matter
Learning rate × number of trees first; then leaves / depth; then regularization (`l2_regularization`, `min_samples_leaf`). A lower rate with more trees is smoother; too high a rate overshoots.

In [ ]:
for lr in [0.5, 0.1, 0.03]:
    m = HistGradientBoostingClassifier(learning_rate=lr, max_iter=3000, early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, categorical_features='from_dtype', random_state=835).fit(X_tr, y_tr)
    print(f'learning_rate {lr:<5}: stopped at {m.n_iter_:4d} trees   test AUC {roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.4f}')

## 4. LightGBM and XGBoost — same idea, different defaults

In [ ]:
# OPTIONAL — pip install lightgbm xgboost (already installed on Colab)
import lightgbm as lgb, xgboost as xgb
lg = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8, verbose=-1, random_state=835)
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=835)
lg.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
print(f'LightGBM: {lg.best_iteration_} trees   test AUC {roc_auc_score(y_te, lg.predict_proba(X_te)[:, 1]):.4f}')
xg = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, enable_categorical=True, tree_method='hist', early_stopping_rounds=50, random_state=835)
xg.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
print(f'XGBoost:  {xg.best_iteration} trees   test AUC {roc_auc_score(y_te, xg.predict_proba(X_te)[:, 1]):.4f}')

## 5. The tabular frontier: GBDT vs. foundation models
2022: tree ensembles beat neural nets on tabular data (Grinsztajn et al.). 2025: **TabPFN-2.5** wins against default XGBoost on nearly every dataset under 10k rows, zero tuning. 2026: **TabICLv2** (Inria, open source, sklearn-compatible) does it at up to 500k rows. On a 5,000-row slice, compare.

In [ ]:
# OPTIONAL — Colab: !pip install -q tabicl   (downloads pretrained weights on first use; CPU works, GPU is faster)
Xs = X_tr.sample(5000, random_state=835); ys = y_tr.loc[Xs.index]
small_gbm = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=3000, early_stopping=True, categorical_features='from_dtype', random_state=835).fit(Xs, ys)
print(f'GBDT trained on 5,000 rows: test AUC {roc_auc_score(y_te, small_gbm.predict_proba(X_te)[:, 1]):.4f}')
try:
    from tabicl import TabICLClassifier
    Xs_num = Xs.apply(lambda c: c.cat.codes if hasattr(c, 'cat') else c); Xte_num = X_te.apply(lambda c: c.cat.codes if hasattr(c, 'cat') else c)
    t0 = time.time(); tab = TabICLClassifier().fit(Xs_num, ys)
    print(f'TabICL, zero tuning:        test AUC {roc_auc_score(y_te, tab.predict_proba(Xte_num)[:, 1]):.4f}   ({time.time()-t0:.0f}s)')
except ImportError:
    print('tabicl not installed — run the pip line above in Colab')

In [ ]:
# OPTIONAL — AutoML in three lines (Colab: !pip install -q autogluon.tabular). AutoGluon 1.4+ bundles GBDTs, TabPFN, TabICL and stacks them.
try:
    from autogluon.tabular import TabularPredictor
    tr_df = X_tr.assign(is_canceled=y_tr.values).sample(20000, random_state=835)
    pred = TabularPredictor(label='is_canceled', eval_metric='roc_auc', verbosity=0).fit(tr_df, time_limit=300, presets='medium_quality')
    print(pred.leaderboard(X_te.assign(is_canceled=y_te.values), silent=True)[['model', 'score_test']].head(6))
except ImportError:
    print('autogluon not installed — see the pip line above (Colab only; ~2 min install)')

**Reading the result:** on small data a pretrained tabular transformer with zero hyperparameters typically matches the boosted model you just tuned. On 95,000 rows the GBDT pulls ahead and runs in production with no license question (TabPFN-3 weights are non-commercial; TabICL is open). The 2026 answer is *both*: GBDT as the workhorse, a TFM as the instant baseline.

## 6. Competition reveal & what worked
The pattern every year: clean pipelines beat clever models; early-stopped boosting sits at the top; the teams that tuned against the public leaderboard fall on the private one.

## 7. Your turn
1. **Regularize.** Add `l2_regularization=1.0` and `min_samples_leaf=50` to the HistGB model. Does test AUC move? Does the stopping iteration?
2. **Monotone constraint.** Force the effect of `lead_time` to be non-decreasing (`monotonic_cst`). Does it cost accuracy? Why might a revenue manager want it anyway?
3. **Regression.** Use `HistGradientBoostingRegressor` on the Session 4 Ames pipeline and compare RMSE with lasso.

In [ ]:
# Your turn — work here

## What we learned tonight
- **Boosting fits the residuals sequentially** with shallow trees and a small learning rate — bias reduction, where bagging is variance reduction.
- **Early stopping on a validation set** is the one habit that separates a boosting model that generalizes from one that memorized.
- **Tree ensembles still beat neural nets on tabular data; tabular foundation models now match tuned boosting on small/medium tables** with zero tuning. GBDT for scale, latency, licensing; TFM as the fast baseline.

Memo due tonight. **HW4** assigned next week (due Nov 23).